# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sinagdevv/flyrank-ml-internship-sinagdev/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

I choose the official **Refresh / Content Opportunity Scoring** lane. It asks which pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring. It fits the starter data because each row is a pseudonymized content item with observable search, engagement, age, freshness, and trend fields. The provisional output is an interpretable ranked review queue for a human, not an automatic edit decision. A later version should use a future-window outcome from the warehouse where the data supports it.

In [1]:
decision_frame = {
    "lane": "Refresh / Content Opportunity Scoring",
    "output": "ranked pages for human review",
    "action": "refresh, expand, protect, prune, or monitor after review",
}
assert decision_frame["lane"] == "Refresh / Content Opportunity Scoring"
print("Frame check passed:", decision_frame["output"], "with an explicit human action.")

Frame check passed: ranked pages for human review with an explicit human action.


## 2. The question: decision, action, cost of a wrong call

**Research question:** Among pages with enough exposure to matter, can an interpretable score rank pages so the top-K review queue contains more observed decline cases than a transparent stale-and-visible baseline?

The decision-maker is a content strategist or SEO editor deciding which page to inspect first. The action is a human review that may lead to refresh, expansion, protection, pruning, or monitoring; the score does not authorize an edit by itself. The unit of analysis is one pseudonymized content item/page, with client IDs used only for grouped validation.

For this starter exercise, the provisional measured outcome is `trend_direction == "down"`, and the primary success metric is Precision@K (especially Precision@50) evaluated with a client-holdout split. This is a current observed screening outcome, not a future recovery label: `trend_direction` is derived from `trend_pct` and must not be used as a feature. A wrong positive recommendation costs scarce editor time and opportunity cost; a wrong negative recommendation can leave a genuinely weak, high-exposure page unreviewed. Data or ML may help because several observable signals—exposure, freshness, position, CTR, engagement, age, and content structure—can interact in ways that a single stale-page rule cannot capture. The rule remains the baseline, and extra model complexity must earn its place on the decision metric.

In [2]:
candidate_signals = {"impressions_90d", "days_since_last_update", "avg_position", "ctr", "engagement_rate", "content_type"}
forbidden_as_features = {"content_id", "client_id", "trend_direction", "trend_pct"}
assert not candidate_signals & forbidden_as_features
print("Decision check passed: ranking uses observable candidate signals; IDs and current outcome fields stay out of features.")

Decision check passed: ranking uses observable candidate signals; IDs and current outcome fields stay out of features.


## 3. Quick look at the data (2-3 real numbers)

The next cell recomputes public-safe aggregate measurements from the authorized starter CSV. The provisional editorial cutoffs are impressions_90d >= 500 for visibility and days_since_last_update >= 180 for staleness; neither threshold is optimized or universal. The stale-visible comparison is useful evidence to test, but its small stale group must keep the claim directional.

**Reconciliation note:** ML-01's reported 27,532 was not a different dataset or pipeline output. Its code selected the search_volume and impressions_90d columns and then called .dropna(), so it excluded rows with missing search_volume. This notebook uses only impressions_90d > 0 for its eligibility count; the code below reproduces both definitions.

In [3]:
from pathlib import Path
import pandas as pd

repo_root = Path.cwd()
if not (repo_root / "data/raw/content_refresh_anonymized.csv").exists():
    repo_root = (repo_root / "../..").resolve()
data_path = repo_root / "data/raw/content_refresh_anonymized.csv"
assert data_path.exists(), f"Starter CSV not found: {data_path}"
df = pd.read_csv(data_path)
required = {"content_id", "client_id", "search_volume", "impressions_90d", "days_since_last_update", "trend_direction"}
assert required <= set(df.columns)
required_nonmissing = required - {"search_volume"}
assert df[list(required_nonmissing)].notna().all().all()

impressions_eligible = df["impressions_90d"].gt(0)
ml01_filter_reproduction = df.loc[impressions_eligible, ["search_volume", "impressions_90d"]].dropna()
assert len(ml01_filter_reproduction) == int(df["search_volume"].notna().sum())
declining_now = df["trend_direction"].astype("string").str.lower().eq("down")
visible = df["impressions_90d"].ge(500)
stale_visible = visible & df["days_since_last_update"].ge(180)
fresh_visible = visible & ~df["days_since_last_update"].ge(180)

print(f"Starter slice: {len(df):,} pages across {df['client_id'].nunique():,} clients.")
print(f"Eligibility reconciliation: impressions_90d > 0 = {int(impressions_eligible.sum()):,}; ML-01 plus search_volume.dropna() = {len(ml01_filter_reproduction):,}; excluded for missing search_volume = {int(df['search_volume'].isna().sum()):,}.")
print(f"Visible pages (>=500 impressions_90d): {int(visible.sum()):,} ({visible.mean():.1%}).")
print(f"Stale + visible: {int(stale_visible.sum()):,} pages; observed down rate {declining_now[stale_visible].mean():.1%}.")
print(f"Fresh + visible: {int(fresh_visible.sum()):,} pages; observed down rate {declining_now[fresh_visible].mean():.1%}.")

Starter slice: 30,000 pages across 32 clients.
Eligibility reconciliation: impressions_90d > 0 = 30,000; ML-01 plus search_volume.dropna() = 27,532; excluded for missing search_volume = 2,468.
Visible pages (>=500 impressions_90d): 16,726 (55.8%).
Stale + visible: 17 pages; observed down rate 94.1%.
Fresh + visible: 16,709 pages; observed down rate 59.5%.


## 4. Careful words: what I can and can't claim

I can say that the starter slice contains a measurable review inventory, that visible pages are common enough to rank, and that the current stale-visible group has a different observed down rate from fresh-visible pages. Those are measured, directional associations that justify testing a ranked review queue.

I cannot say that a refresh will cause recovery, that the current `trend_direction` outcome predicts a future window, or that the analysis predicts Google’s algorithm. The stale-visible result is based on only one small group in one 90-day snapshot, so it may be unstable. Later work must define a leakage-safe future target, keep IDs and outcome-derived fields out of features, validate by client and/or time as appropriate, compare against the transparent baseline, and report Precision@K with reason codes. Human review remains part of the decision.

In [4]:
claims = {
    "observed": True,
    "directional": True,
    "decision_support": True,
    "causal_proof": False,
    "predicts_google": False,
}
assert claims["observed"] and claims["directional"] and claims["decision_support"]
assert not claims["causal_proof"] and not claims["predicts_google"]
print("Claim-boundary check passed: observed, directional, decision-support only.")

Claim-boundary check passed: observed, directional, decision-support only.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] IDs and outcome-derived fields are excluded from candidate features
- [ ] Committed to my repo under `work/notebooks/` — intentionally deferred until review.